# 🚀 Фабрика нейросетей & Асинхронный Walk-Forward рой PRL/USDT (Google Colab / GPU T4)
Весь тяжёлый код архитектур и обучения вынесен в модульные скрипты  и .
Управляйте параметрами обучения напрямую через аргументы в ячейках ниже.

In [1]:
# 1. Инициализация окружения Colab и принудительное обновление скриптов из репозитория
!nvidia-smi
import os, sys, importlib
if os.path.exists("/content"):
    !rm -rf _repo_tmp && git clone --depth 1 https://github.com/BulletproofBoris/crypto_bot.git _repo_tmp 2>/dev/null || true
    !mkdir -p _tools && cp -rf _repo_tmp/_tools/* _tools/
sys.path.insert(0, os.path.abspath("."))
import _tools.colab_models, _tools.colab_neural_trainer
importlib.reload(_tools.colab_models)
importlib.reload(_tools.colab_neural_trainer)
from _tools.colab_neural_trainer import run_colab_pipeline
print("✅ Свежие модули _tools.colab_models и _tools.colab_neural_trainer загружены!")

Wed Oct 21 09:36:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

✅ Свежие модули _tools.colab_models и _tools.colab_neural_trainer загружены!


In [ ]:
# 2. Конфигурация и запуск Walk-Forward обучения нейросетей (редактируемые параметры)
# Асинхронная сетка окон: H=25 -> 2 окна (375k), H=12 -> 5 окон (210k), H=5 -> 12 окон (75k)
HORIZON = 25                                               # Горизонт H (сек): 5, 12 или 25
WINDOWS = "0,1"                                            # Список окон: "all" или "0,1"
ARCHS = "conv1d+gru,causal_cnn,attention,mlp_bottleneck"   # Архитектуры через запятую
RUNS_PER_ARCH = 15                                         # Число независимых ранов на архитектуру
MAX_EPOCHS = 10                                            # Макс. эпох на один ран
BATCH_SIZE = 2048                                          # Размер батча на GPU T4
LR = 1e-2                                                  # Скорость обучения AdamW

df_leaderboard = run_colab_pipeline(
    horizon=HORIZON, windows=WINDOWS, archs=ARCHS,
    runs_per_arch=RUNS_PER_ARCH, max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE, lr=LR
)

☁️ Прямое скачивание prl_neural_colab_data.npz с Google Drive (ID=1IXq51Irot9L-pj8-EVAY980J7tql4xho)...
⚡ Подготовка memory-mapped кэша на диске (экономит ~4 ГБ RAM)...

🚀 ОБУЧЕНИЕ ОКНА WIN_00 | H=25с (K=3.0, L=75с) | Train=375,000 (2026-10-14 15:21..2026-10-20 08:32)
W0000 00:00:1792575434.658368    1452 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: 0.005000
   ⏳ [Elastic EarlyStop] Достигнут эластичный потолок (7/10 эп.).
  [CONV1D+GRU | win_00 | r01/15] 🏆 РЕКОРД (212.5с) -> Loss: 1.5099 | IC: +0.2071 | AUC: 0.7433 | Top-2%: +0.16
   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: 0.005000
   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: 0.002500
   ⏳ [Elastic EarlyStop] Достигнут эластичный потолок (7/10 эп.).
  [CONV1D+GRU | win_00 | r02/15] 🏆 РЕКОРД (208.8с

: 

In [2]:
# 3. Выгрузка готового архива моделей (проверенные зеркала x0.at + gofile + filebin)
import os
HORIZON = globals().get("HORIZON", 25)
ZIP_NAME = f"prl_ensemble_pack_h{HORIZON}s.zip"
if os.path.exists(ZIP_NAME):
    print(f"📦 Архив найден: {ZIP_NAME} ({os.path.getsize(ZIP_NAME) / (1024*1024):.2f} MB)")
    !curl -s --max-time 60 -F "file=@{ZIP_NAME}" https://x0.at/ | head -c 200 && echo " (x0.at)"
    !curl -s --max-time 60 -F "file=@{ZIP_NAME}" https://store1.gofile.io/uploadFile | head -c 400 && echo " (gofile)"
    !curl -s --max-time 60 -H "Accept: application/json" -T "{ZIP_NAME}" https://filebin.net/prl_h{HORIZON}s_pack/{ZIP_NAME} | head -c 300 && echo " (filebin)"

📦 Архив найден: prl_ensemble_pack_h25s.zip (5.38 MB)
https://x0.at/cLum.zip
 (x0.at)
{"data":{"code":"8V8PudOt","createTime":1792585631,"downloadPage":"https://gofile.io/d/595Jhj4A","guestToken":"iMvTrDwXJXvAlw51ANn0xh0RjjiMxrOF","id":"e4d5fa6d-0e47-4241-b6d0-4e00100401da","md5":"40c74d5f68532ced2a730bbc4a9ccca4","mimetype":"application/zip","modTime":1792585631,"name":"prl_ensemble_pack_h25s.zip","parentFolder":"8d78d4bc-65b1-4089-8e0f-03674df485d1","parentFolderCode":"595Jhj4A", (gofile)
{
    "bin": {
        "id": "prl_h25s_pack",
        "readonly": false,
        "bytes": 5644404,
        "bytes_readable": "5.6 MB",
        "files": 1,
        "updated_at": "2026-10-21T12:27:14.689982Z",
        "updated_at_relative": "now",
        "created_at": "2026-10-21T12:17:32.137515Z",
  (filebin)
